In [1]:
from huggingface_hub import login
import os
from dotenv import load_dotenv

load_dotenv()
login(token=os.getenv("HUGGINGFACE_TOKEN"))

In [2]:
from datasets import load_dataset

# Stream one English sample — no full download
ds = load_dataset(
    "amphion/Emilia-Dataset",
    split="train",
    streaming=True,
)
sample = next(iter(ds))

print("Keys:", list(sample.keys()))
print("Text:", sample.get("text", sample.get("json", {}).get("text", "—")))

audio = sample["mp3"]  # dict with 'array' and 'sampling_rate'
print(
    f"Sample rate: {audio['sampling_rate']} Hz, length: {len(audio['array'])} samples"
)

Resolving data files:   0%|          | 0/4343 [00:00<?, ?it/s]

Keys: ['json', 'mp3', '__key__', '__url__']
Text:  So. Chloe hat gesagt, ich soll noch unten gehen. Was ich natürlich auch machen werde.
Sample rate: 24000 Hz, length: 193968 samples


In [3]:
import torch

# transform audio to tensor and add dimension
audio_tensor = torch.tensor(audio["array"]).unsqueeze(0).float()
sr = audio["sampling_rate"]
print(f"Audio tensor: {audio_tensor.shape}, sr={sr}")

Audio tensor: torch.Size([1, 193968]), sr=24000


In [4]:
from TTS.api import TTS

# bypass coqui aggrement
os.environ["COQUI_TOS_AGREED"] = "1"
# Downloads and caches to ~/.local/share/tts/ on first run
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2")
model = tts.synthesizer.tts_model
print("Model loaded:", type(model).__name__)

C:\Users\Andre\PycharmProjects\Speech-technology-project-7\.venv\Lib\site-packages\jsonlines\jsonlines.py:324: SyntaxWarning: invalid escape sequence '\*'
  :param \*\*kwargs: additional arguments, forwarded to the reader or writer
C:\Users\Andre\PycharmProjects\Speech-technology-project-7\.venv\Lib\site-packages\pysbd\segmenter.py:66: SyntaxWarning: invalid escape sequence '\s'
  for match in re.finditer('{0}\s*'.format(re.escape(sent)), self.original_text):
C:\Users\Andre\PycharmProjects\Speech-technology-project-7\.venv\Lib\site-packages\pysbd\lang\arabic.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)
C:\Users\Andre\PycharmProjects\Speech-technology-project-7\.venv\Lib\site-packages\pysbd\lang\persian.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)


Model loaded: Xtts


In [5]:
gpt_cond_latent = model.get_gpt_cond_latents(
    audio_tensor, sr, length=model.config.gpt_cond_len
)
speaker_embedding = model.get_speaker_embedding(audio_tensor, sr)

print("gpt_cond_latent shape:", gpt_cond_latent.shape)
print("speaker_embedding shape:", speaker_embedding.shape)

gpt_cond_latent shape: torch.Size([1, 32, 1024])
speaker_embedding shape: torch.Size([1, 512, 1])


---

# Modifying each dimension individually

In [6]:
import ipywidgets as widgets
from IPython.display import Audio, display, clear_output

# --- Setup State ---
working_gpt_latent = gpt_cond_latent.clone()
working_spk_emb = speaker_embedding.clone()

# --- UI Components ---
output_area = widgets.Output()
dimension_window = 20 # Number of sliders to show at once

# Select which embedding to tweak
emb_selector = widgets.Dropdown(
    options=[('GPT Latent (Style/Prosody)', 'gpt'), ('Speaker Embedding (Tone/Identity)', 'spk')],
    value='gpt',
    description='Target:'
)

# Window slider (since we can't show 1024 sliders at once)
window_slider = widgets.IntSlider(
    value=0, min=0, max=1024-dimension_window, step=1,
    description='Dim Offset:', continuous_update=False
)

slider_container = widgets.VBox([])

def create_sliders(change=None):
    """Refreshes the sliders based on the selected embedding and window."""
    target = emb_selector.value
    offset = window_slider.value

    # Update window max based on selection
    window_slider.max = (1024 if target == 'gpt' else 512) - dimension_window

    current_data = working_gpt_latent[0, 0] if target == 'gpt' else working_spk_emb[0]

    sliders = []
    for i in range(offset, offset + dimension_window):
        val = current_data[i].item()
        # Create slider with a range around the current value
        s = widgets.FloatSlider(
            value=val, min=val-2.0, max=val+2.0, step=0.01,
            description=f"Dim {i}", continuous_update=False,
            layout=widgets.Layout(width='90%')
        )
        s.observe(lambda change, idx=i: update_value(change, idx, target), names='value')
        sliders.append(s)

    slider_container.children = sliders

def update_value(change, idx, target):
    """Updates the underlying tensor when a slider moves."""
    if target == 'gpt':
        working_gpt_latent[0, 0, idx] = change['new']
    else:
        working_spk_emb[0, idx] = change['new']

def run_inference(b):
    """Triggers the XTTS synthesis with modified embeddings."""
    with output_area:
        clear_output()
        print("Synthesizing with modified embeddings... 🎙️")

        # Note: Speaker embeddings are usually L2-normalized in XTTS
        norm_spk = torch.nn.functional.normalize(working_spk_emb, p=2, dim=1)

        out = model.inference(
            text="This is how I sound after you modified my latent dimensions.",
            language="en",
            gpt_cond_latent=working_gpt_latent,
            speaker_embedding=norm_spk,
            temperature=model.config.temperature,
            length_penalty=model.config.length_penalty,
            repetition_penalty=model.config.repetition_penalty,
            top_k=model.config.top_k,
            top_p=model.config.top_p,
        )

        display(Audio(out['wav'], rate=24000, autoplay=True))
        print(f"L2 Norm of Speaker Emb: {torch.norm(norm_spk).item():.4f}")

# --- Event Listeners ---
emb_selector.observe(create_sliders, names='value')
window_slider.observe(create_sliders, names='value')

generate_btn = widgets.Button(description="Generate & Play", button_style='success', icon='play')
generate_btn.on_click(run_inference)

reset_btn = widgets.Button(description="Reset Embeddings", button_style='danger')
def reset_values(b):
    global working_gpt_latent, working_spk_emb
    working_gpt_latent = gpt_cond_latent.clone()
    working_spk_emb = speaker_embedding.clone()
    create_sliders()
reset_btn.on_click(reset_values)

# --- Layout ---
create_sliders() # Initial call
ui = widgets.VBox([
    widgets.HTML("<h2>XTTS Latent Space Explorer</h2>"),
    widgets.HBox([emb_selector, window_slider]),
    widgets.HBox([generate_btn, reset_btn]),
    widgets.HTML("<br><b>Adjusting Dimensions (Current Window):</b>"),
    slider_container,
    output_area
])

display(ui)